In [ ]:
begin
    using Pkg: Pkg
    # Activates the notebook environment
    Pkg.activate(@__DIR__)
    # Ensures all dependencies in Project.toml are installed
    Pkg.instantiate()
end



In [ ]:
begin
    const SCRIPT_DIR  = @__DIR__
    const FIGURES_DIR = joinpath(SCRIPT_DIR, "figures", "main", "elephant_movement")
    mkpath(FIGURES_DIR)

    ENV["GKS_ENCODING"] = "utf8"
    using CSV
    using DataFrames
    using Dates
    using StateSpaceDynamics
    using StateSpaceDynamics.ACDC
    using LinearAlgebra
    using Random
    using Serialization
    using Distributions
    using Plots, Measures

    include(joinpath(SCRIPT_DIR, "helpers", "AnimalMovementUtils.jl"))
    using .AnimalMovementUtils
end

In [ ]:
begin
    # Load the multi-taxa GPS file (Movebank movement-syndrome dataset)
    df_raw = CSV.read(joinpath(SCRIPT_DIR, "data", "movement-syndrome-vertebrate-taxa-gps.csv"), DataFrame)
    println("Columns: ", names(df_raw))
    println("Rows: ", nrow(df_raw))
    println("Individuals: ", length(unique(df_raw[:, "individual-local-identifier"])))
    println("Taxa: ", unique(df_raw[:, "individual-taxon-canonical-name"]))
end

In [ ]:
df_clean = prepare_movement_data(df_raw);

In [ ]:
individual_id, animal_data = get_most_observed_individual(df_clean)

In [ ]:
begin
    steps = disallowmissing(animal_data[:, :step_length])
    println("Timesteps:      ", nrow(animal_data))
    println("Step median:    ", round(median(steps); digits=1), " m")
    println("Step max:       ", round(maximum(steps); digits=1), " m")
    println("Date range:     ",
        animal_data[1, "timestamp"], " → ", animal_data[end, "timestamp"])
end

## Model definitions

`StepAngleEmission`, `haversine_dist`, `prepare_movement_data` and all fitting/plotting
utilities are defined in `AnimalMovementUtils.jl`.

Three model variants are compared:
- **Base**: `StepAngleEmission` — Gaussian on z-scored log step length + WrappedCauchy on turn angle
- **Alternative**: `GaussianEmission` on `[z_log_step, turn_angle]`
- **Cardioid**: Gaussian on z-scored log step length + Cardioid on turn angle


In [ ]:
analysis_results = compute_movement_analysis(
    animal_data, individual_id, 42;
    K_range   = 1:8,
    use_cache = true,
)
println("BIC selects K=", analysis_results.selections.bic)
println("ICL selects K=", analysis_results.selections.icl)

In [ ]:
analysis_results_alt = compute_movement_analysis_alt(
    animal_data, individual_id, 42;
    K_range   = 1:8,
    use_cache = true,
)
println("BIC selects K=", analysis_results_alt.selections.bic)
println("ICL selects K=", analysis_results_alt.selections.icl)

In [ ]:
analysis_results_alt2 = compute_movement_analysis_alt2(
    animal_data, individual_id, 42;
    K_range   = 1:8,
    use_cache = true,
)
println("BIC selects K=", analysis_results_alt2.selections.bic)
println("ICL selects K=", analysis_results_alt2.selections.icl)

In [ ]:
movement_plots = get_movement_plots(analysis_results, individual_id; figures_dir=FIGURES_DIR, K_display=3)

In [ ]:
movement_plots_alt = get_movement_plots_alt(analysis_results_alt, individual_id; figures_dir=FIGURES_DIR, K_display=3)

In [ ]:
movement_plots_alt2 = get_movement_plots_alt2(analysis_results_alt2, individual_id; figures_dir=FIGURES_DIR, K_display=3)

In [ ]:
begin
    stats_path = joinpath(SCRIPT_DIR, "results", "movement_statistics.txt")
    write_movement_statistics(stats_path, analysis_results, analysis_results_alt,
        "[alt2] StepAngleCardEmission (Gaussian log-step + Cardioid angle)" => analysis_results_alt2)
end

In [ ]:
movement_plots.trajectory

In [ ]:
movement_plots.selection

In [ ]:
movement_plots_alt.selection

In [ ]:
movement_plots_alt2.selection

In [ ]:
begin
    base_xlims   = feature_xlims_from_res(analysis_results)
    alt1_xlims   = feature_xlims_from_res(analysis_results_alt)
    alt2_xlims   = feature_xlims_from_res(analysis_results_alt2)

    for K_show in (2, 3, 4, 8)
        p_base = movement_state_feature_grid(
            analysis_results, K_show;
            feature_names = ["z_log_step", "turn_angle"],
            feature_xlims = base_xlims,
            color = :steelblue,
        )
        AnimalMovementUtils._save_png_pdf(p_base, joinpath(SCRIPT_DIR, "figures", K_show == 3 ? "main" : "supplementary", "elephant_movement"), "movement_state_feature_grid_base_k$(K_show)")

        p_alt1 = movement_state_feature_grid(
            analysis_results_alt, K_show;
            feature_names = ["z_log_step", "turn_angle"],
            feature_xlims = alt1_xlims,
            color = :darkorange,
        )
        AnimalMovementUtils._save_png_pdf(p_alt1, joinpath(SCRIPT_DIR, "figures", K_show == 3 ? "main" : "supplementary", "elephant_movement"), "movement_state_feature_grid_alt1_k$(K_show)")

        p_alt2 = movement_state_feature_grid(
            analysis_results_alt2, K_show;
            feature_names = ["z_log_step", "turn_angle"],
            feature_xlims = alt2_xlims,
            color = :seagreen,
        )
        AnimalMovementUtils._save_png_pdf(p_alt2, joinpath(SCRIPT_DIR, "figures", K_show == 3 ? "main" : "supplementary", "elephant_movement"), "movement_state_feature_grid_alt2_k$(K_show)")
    end
end